# Replicate the causal patch Transformer across training seeds

Does the seed-42 architecture gain repeat with seeds 43 and 44?
Reuse the completed patch candidate for seed 42 and every measured dropout-0.2 reference.
Train only two new patch candidates across five development folds each: ten fits, at most 400 epochs.

Keep patch size three, raw context thirty, width sixteen, dropout 0.2, constant Adam 0.001, and budget 40/12 fixed.
Keep features, masks, scaling, natural sampling, unweighted loss, diagnostics, and macro-F1 checkpoint selection fixed.
The model remains a joint-channel causal patch classifier, not a faithful PatchTST forecasting reproduction.

The same 26 development participants repeat across seeds. The three final-test participants remain reserved.
Training seeds change initialization, batch order, and dropout randomness; split seed 42 stays fixed.
These are development robustness checks, not independent participant or final-test estimates.
The seed-42 cold-start loss and essentially flat transition score remain explicit follow-up concerns.
Do not change partial-patch handling or another architecture setting during this replication.
Read docs/patch-transformer-seeds.md before enabling preparation or training.

Completed results: docs/experiments/patch-transformer-seeds-2026-11-13/README.md


## Locate the project and set execution gates

**What:** Find the repository, configure logging, and declare optional execution switches.
**Why:** Use the same paths across local and cloned environments while keeping model optimization explicitly disabled.


In [ ]:
import json
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("patch_transformer_seed_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/transformer-patch-seeds.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()
REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "experiment/patch-transformer-seeds"  # Select the PR branch before merge
RUN_SETUP = False
RUN_CHECK = True
RUN_PREPARATION = False
RUN_TRAINING = False
logger.info("Repository: %s", ROOT)
logger.info("Run setup: %s", RUN_SETUP)
logger.info("Run check: %s", RUN_CHECK)
logger.info("Run preparation: %s", RUN_PREPARATION)
logger.info("Run training: %s", RUN_TRAINING)

## Optional environment setup

**What:** Clone the working branch if absent, install uv through Astral if necessary, and install locked dependencies.
**Why:** Make a fresh machine reproducible while allowing the existing local environment to skip setup.


In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    
    %cd "{ROOT}"
    !uv sync --locked --group gru --group notebook --group dev
    
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")
else:
    logger.info("Environment setup skipped")

## Inspect new work and verify reused results

**What:** Load the explicit seed pairs, validate completed reused artifacts, and display the work budget.
**Why:** Retrain only the two new candidates and refuse missing or damaged references instead of silently replacing them.


In [ ]:
import pandas as pd

from sleep_classifier.grouped_validation import GroupedValidationExperiment
from sleep_classifier.neural_training import TrainingConfig
from sleep_classifier.paired_architecture_report import PairedArchitectureReport
from sleep_classifier.paired_seed_report import PairedSeedReport

study = PairedSeedReport(ROOT / "configs/transformer-patch-seeds.yaml")
plan = study.plan()
study.validate_reused_results()
new_runs = study.new_runs()

if [(run["seed"], run["recipe"]) for run in new_runs] != [(43, "candidate"), (44, "candidate")]:
    raise ValueError("Only patch candidates for seeds 43 and 44 may require new fits")
if plan["new_fits"] != 10 or plan["maximum_new_epochs"] != 400:
    raise ValueError("The seed study exceeds the agreed work budget")

display(pd.DataFrame(plan["cases"]).set_index("seed"))
display(pd.DataFrame(new_runs)[["seed", "recipe", "folds", "maximum_epochs"]])

logger.info("Reused artifacts verified; new fits=%s maximum epochs=%s",
            plan["new_fits"], plan["maximum_new_epochs"])
logger.info("New runs: %s", [(run["seed"], run["recipe"]) for run in new_runs])

## Verify frozen controls, fold assignments, and devices

**What:** Compare every grouped plan and each role's recipe across all three training seeds.
Also compare reused recipes with their saved fold metrics and resolved device.
**Why:** A different split, architecture, observation control, or device would invalidate the intended initialization comparison.


In [ ]:
expected_folds = None
canonical_recipes = {}
expected_device = None
for case in study.cases:
    for role in ("reference", "candidate"):
        experiment = GroupedValidationExperiment(Path(case[f"{role}_config"]))
        grouped_plan = experiment.plan()
        if len(grouped_plan["folds"]) != 5 or len(grouped_plan["development"]) != 26:
            raise ValueError("Each recipe must use all five development folds")
        if len(grouped_plan["reserved_test"]) != 3 or grouped_plan["split_seed"] != 42:
            raise ValueError("The frozen split or reserved test membership changed")
        if expected_folds is None:
            expected_folds = grouped_plan["folds"]
        if grouped_plan["folds"] != expected_folds:
            raise ValueError("Seed configurations use different participant assignments")

        recipe = {key: value for key, value in experiment.model_settings.items()
                  if key not in {"seed", "output_dir"}}
        if role not in canonical_recipes:
            canonical_recipes[role] = recipe
        if recipe != canonical_recipes[role]:
            raise ValueError("A model or training setting differs across seeds")
        if (role == "candidate" and (recipe["model_type"] != "patch_transformer"
                                     or recipe["patch_size"] != 3)):
            raise ValueError("The candidate patch architecture changed")

        resolved_device = str(TrainingConfig(experiment.model_settings).device())
        if expected_device is None:
            expected_device = resolved_device
        if resolved_device != expected_device:
            raise ValueError("Configured devices differ across the study")

        if case[f"reuse_{role}"]:
            summary_path = Path(case[f"{role}_summary"])
            saved = json.loads(summary_path.read_text())
            if saved["plan"] != grouped_plan:
                raise ValueError("A reused result differs from its configured fold plan")
            manifest = json.loads((summary_path.parent / "manifest.json").read_text())
            if manifest["device"] != resolved_device:
                raise ValueError("Current device differs from the measured reference device")
            for result in saved["results"]:
                path = Path(result["metrics_path"])
                if not path.is_absolute():
                    path = summary_path.parent / path
                measured = json.loads(path.read_text())["config"]
                excluded = {"seed", "prepare_config", "output_dir", "device"}
                measured_recipe = {key: value for key, value in measured.items()
                                   if key not in excluded}
                configured_recipe = {key: value for key, value in recipe.items()
                                     if key not in excluded}
                if measured_recipe != configured_recipe:
                    raise ValueError("A reused recipe differs from its measured settings")

logger.info("All six configurations retain the original folds, fixed controls, and device=%s",
            expected_device)

## Prepare only new candidate folds

**What:** Optionally freeze inputs and configurations for patch seeds 43 and 44.
**Why:** Reuse completed references and the seed-42 candidate while keeping prepared files immutable.
Preparation creates no optimizer updates. Existing output directories require inspection rather than automatic replacement.


In [ ]:
if RUN_PREPARATION:
    study.validate_reused_results()
    %cd "{ROOT}"
    for run in study.new_runs():
        grouped_config = run["config"]
        logger.info("Preparing patch candidate seed %s", run["seed"])
        !uv run --locked --group gru --group notebook python \
            -m sleep_classifier.grouped_validation --config "{grouped_config}" --prepare
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Preparation failed; inspect retained artifacts before retrying")
else:
    logger.info("Candidate preparation skipped")

## Check the actual prepared candidate folds without training

**What:** Run a no-gradient shape and device check for each newly prepared candidate fold.
**Why:** Validate the exact ten fold configurations before allowing optimization; report unprepared candidates as pending.


In [ ]:
if RUN_CHECK:
    %cd "{ROOT}"
    for run in study.new_runs():
        output_dir = Path(run["summary"]).parent
        manifest_path = output_dir / "manifest.json"
        
        if not manifest_path.exists():
            logger.info("Seed %s preflight pending: prepare its candidate first", run["seed"])
            continue
        
        manifest = json.loads(manifest_path.read_text())
        for fold in manifest["plan"]["folds"]:
            fold_config = output_dir / fold["name"] / "gru.yaml"
            logger.info("Checking seed %s %s without optimization", run["seed"], fold["name"])
            
            !uv run --locked --group gru --group notebook python \
                -m sleep_classifier.neural_training --config "{fold_config}" --check
            
            if get_ipython().user_ns["_exit_code"] != 0:
                raise RuntimeError("Patch seed preflight failed")
else:
    logger.info("Candidate preflights skipped")

## Train the two new grouped candidates when ready

**What:** Optionally train seed 43 and then seed 44, each with five fresh models and training-only scalers.
**Why:** Measure training-randomness effects with the same observations and controls.
A failed command stops the loop. Automatic resume is unsupported; inspect completed artifacts before choosing recovery.


In [ ]:
if RUN_TRAINING:
    study.validate_reused_results()
    %cd "{ROOT}"
    for run in study.new_runs():
        grouped_config = run["config"]
        logger.info("Training patch candidate seed %s", run["seed"])
        !uv run --locked --group gru --group notebook python -u \
            -m sleep_classifier.grouped_validation --config "{grouped_config}" --train
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Training failed; completed evidence is retained")
else:
    logger.info("Training skipped: enable RUN_TRAINING when ready")

## Inspect complete and pending pairs

**What:** Verify each complete pair and display fold, participant, and stage results.
**Why:** Show the completed seed-42 finding while the new candidates remain pending, without inventing missing results.
Stage indices are 0 = Wake, 1 = Light, 2 = Deep, and 3 = REM.


In [ ]:
statuses = study.status()
display(pd.DataFrame(statuses).set_index(["seed", "recipe"]))

seed_rows = []
class_rows = []
fold_rows = []
for case in study.cases:
    pair_statuses = [row for row in statuses if row["seed"] == case["seed"]]
    if not all(row["status"] == "complete" for row in pair_statuses):
        continue
    paired = PairedArchitectureReport(
        Path(case["reference_summary"]), Path(case["candidate_summary"]),
        comparison=study.comparison,
    ).build()
    reference_mean = pd.DataFrame(paired["folds"])["reference_macro_f1"].mean()
    candidate_mean = pd.DataFrame(paired["folds"])["candidate_macro_f1"].mean()
    seed_rows.append({"seed": case["seed"], "reference_mean_macro_f1": reference_mean,
                      "candidate_mean_macro_f1": candidate_mean,
                      "paired_gain": paired["paired"]["mean_macro_f1_difference"],
                      "participants_improved": sum(row["macro_f1_difference"] > 0
                                                   for row in paired["participants"])})
    for role in ("reference", "candidate"):
        for stage, scores in paired[role]["classes"].items():
            class_rows.append({"seed": case["seed"], "recipe": role,
                               "stage": ["Wake", "Light", "Deep", "REM"][int(stage)], **scores})
    fold_rows.extend({"seed": case["seed"], **row} for row in paired["folds"])

if seed_rows:
    display(pd.DataFrame(seed_rows).set_index("seed"))
    display(pd.DataFrame(class_rows).set_index(["seed", "recipe", "stage"]))
    display(pd.DataFrame(fold_rows).set_index(["seed", "fold"]))

## Build the three-seed aggregate only after completion

**What:** Validate all role-specific recipes and summarize the three paired seed effects.
**Why:** A partial study cannot support the complete mean or sample deviation.
The deviation describes repeated training effects on the same participants; it is not a confidence interval or final-test estimate.


In [ ]:
if all(row["status"] == "complete" for row in statuses):
    result = study.build()
    study.output_dir.mkdir(parents=True, exist_ok=True)
    output = study.output_dir / "summary.json"
    output.write_text(json.dumps(result, indent=2) + "\n")
    display(pd.DataFrame([{
        "mean_seed_paired_gain": result["mean_seed_macro_f1_difference"],
        "sample_std_seed_paired_gain": result["sample_std_seed_macro_f1_difference"],
    }]))
    logger.info("Validated seed comparison saved: %s", output)
else:
    logger.info("Study incomplete: seed-level aggregate requires all three completed pairs")

## Compare overall scores and REM precision, recall, and F1

**What:** Plot mean fold macro-F1 and all three pooled REM scores for each completed pair.
**Why:** A recall gain alone can hide false positives; paired seed effects show whether the original benefit repeats.


In [ ]:
if seed_rows:
    import matplotlib.pyplot as plt

    figure, axes = plt.subplots(1, 4, figsize=(17, 4))
    seed_table = pd.DataFrame(seed_rows).set_index("seed")
    seed_table[["reference_mean_macro_f1", "candidate_mean_macro_f1"]].plot.bar(
        ax=axes[0], ylim=(0, 1),
    )
    axes[0].set(title="Mean fold macro-F1", xlabel="Training seed", ylabel="Score")
    classes = pd.DataFrame(class_rows)
    for axis, metric in zip(axes[1:], ("precision", "recall", "f1"), strict=True):
        rem = classes[classes["stage"] == "REM"].pivot(
            index="seed", columns="recipe", values=metric,
        )
        rem.plot.bar(ax=axis, ylim=(0, 1))
        axis.set(title=f"REM {metric}", xlabel="Training seed", ylabel="Pooled score")
    figure.tight_layout()
    plt.show()

## Inspect every stage's F1

**What:** Plot Wake, Light, Deep, and REM F1 for the completed seed pairs.
**Why:** Confirm that an aggregate gain does not hide losses in another stage.


In [ ]:
if class_rows:
    import matplotlib.pyplot as plt

    classes = pd.DataFrame(class_rows)
    figure, axes = plt.subplots(1, 4, figsize=(17, 4))
    for axis, stage in zip(axes, ("Wake", "Light", "Deep", "REM"), strict=True):
        stage_scores = classes[classes["stage"] == stage].pivot(
            index="seed", columns="recipe", values="f1",
        )
        stage_scores.plot.bar(ax=axis, ylim=(0, 1))
        axis.set(title=stage, xlabel="Training seed", ylabel="Pooled F1")
    figure.tight_layout()
    plt.show()

## Inspect selected-checkpoint probability loss

**What:** Validate saved probabilities and recompute pooled negative log-likelihood for each complete pair.
**Why:** Hard-label scores discard confidence. Lower NLL indicates better average probability loss, but does not establish calibration.
NLL is the mean negative natural logarithm of the true-stage probability; 1e-12 clipping prevents log(0).
Confidence-on-error summaries use changing error sets. Keep their interpretation descriptive.
Scored targets are thirty-second intervals, not training passes.
The seed-42 short-history and transition slices remain exploratory; their differing class supports limit direct comparisons.


In [ ]:
import numpy as np

probability_diagnostics = []
for case in study.cases:
    if not all(row["status"] == "complete" for row in statuses if row["seed"] == case["seed"]):
        continue
    for role in ("reference", "candidate"):
        saved = json.loads(Path(case[f"{role}_summary"]).read_text())
        prediction_frames = []
        for result in saved["results"]:
            path = Path(result["metrics_path"])
            if not path.is_absolute():
                path = Path(case[f"{role}_summary"]).parent / path
            prediction_frames.append(pd.read_csv(path.parent / "validation_predictions.csv"))
        predictions = pd.concat(prediction_frames, ignore_index=True)
        probabilities = predictions[[f"probability_{stage}" for stage in range(4)]].to_numpy()
        if (not np.isfinite(probabilities).all() or (probabilities < 0).any()
                or (probabilities > 1).any()
                or not np.allclose(probabilities.sum(axis=1), 1, atol=1e-6, rtol=0)):
            raise ValueError("Saved class probabilities are invalid")
        
        prediction_column = "prediction" if "prediction" in predictions else "gru_prediction"
        if not np.array_equal(probabilities.argmax(axis=1),
                              predictions[prediction_column].to_numpy(dtype=int)):
            raise ValueError("Saved probabilities disagree with the recorded stage predictions")
        
        targets = predictions.target.to_numpy(dtype=int)
        wrong = probabilities.argmax(axis=1) != targets
        confidence = probabilities.max(axis=1)
        probability_diagnostics.append({
            "seed": case["seed"], "recipe": role, "scored_targets": len(predictions),
            "checkpoint_nll": float(-np.log(np.maximum(
                probabilities[np.arange(len(targets)), targets], 1e-12,
            )).mean()),
            "mean_confidence_correct": float(confidence[~wrong].mean()) if (~wrong).any() else None,
            "mean_confidence_wrong": float(confidence[wrong].mean()) if wrong.any() else None,
        })

if probability_diagnostics:
    display(pd.DataFrame(probability_diagnostics).set_index(["seed", "recipe"]))

## Export HTML without model optimization

**What:** Rebuild this notebook through a fresh kernel and save the executed HTML report.
**Why:** Preserve an output-free source notebook while making existing results reviewable.
Automated export skips setup, preparation, training, and this recursive export cell.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/24_patch_transformer_seeds.ipynb

if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Patch seed report export failed")